In [ ]:
from transformers import DetrForObjectDetection, DetrImageProcessor, TrOCRProcessor, VisionEncoderDecoderModel
import torch
import cv2
import supervision as sv
from PIL import Image
import cv2
from time import sleep

In [ ]:
DETR_CHECKPOINT = "/home/ralvarez22/Adrian/Text2Latex/models/detr/Baizhi/V_16"
TROCR_MODEL = "/home/ralvarez22/Documentos/trocr_hand/trocr_llm/finetuned/Akivili/V_5"
DEVICE = "cuda"
CONFIDENCE_TRESHOLD = 0.5
IOU_TRESHOLD = 0.1

In [ ]:
detr_proc = DetrImageProcessor.from_pretrained(DETR_CHECKPOINT)
detr_model = DetrForObjectDetection.from_pretrained(
    pretrained_model_name_or_path=DETR_CHECKPOINT, 
    ignore_mismatched_sizes=True, local_files_only=True
).to(DEVICE)

#trocr_proc = TrOCRProcessor.from_pretrained(TROCR_MODEL,device_map=DEVICE)
#trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL, device_map=DEVICE)
# Similar to the TROCR Lab, I force the parameters to avoid mistakes
#trocr_model.generation_config.decoder_start_token_id = trocr_proc.tokenizer.bos_token_id
#trocr_model.generation_config.temperature = 0.1

In [ ]:
detr_model.compile()
#trocr_model.compile()

In [ ]:
cam = cv2.VideoCapture(0)

In [ ]:
while True:
    # Read a frame from the camera
    status, photo = cam.read()
    
    #cv2.imshow("Captured Image", photo)
    # Generate all the Bounding boxes
    with torch.no_grad():
        # load image and predict
        inputs = detr_proc(images=photo, return_tensors='pt').to(DEVICE)
        outputs = detr_model(**inputs)

        # post-process
        target_sizes = torch.tensor([photo.shape[:2]]).to(DEVICE)
        results = detr_proc.post_process_object_detection(
            outputs=outputs, 
            threshold=0.1, 
            target_sizes=target_sizes
        )[0]
    
    scores = results["scores"].tolist()
    bboxes = results["boxes"].tolist()
    pil_image = Image.fromarray(photo).convert("RGB")
    
    for box in bboxes:
        start_point = (int(box[0]), int(box[1]))
        end_point = (int(box[2]), int(box[3]))
        #print(start_point, end_point)
        # draw the rectangle
        cv2.rectangle(photo, start_point, end_point, (0, 0, 255), thickness=1, lineType=cv2.LINE_8) 
        # display the output
    
    cv2.imshow('Annotated Image', photo)
    
    #for idx, e in enumerate(bboxes):
    #    crp_img = pil_image.crop(e)
    #    trocr_pixels = trocr_proc(crp_img, return_tensors="pt").pixel_values.to(DEVICE)
    #    trocr_out = trocr_model.generate(trocr_pixels)
    #    rec_text = trocr_proc.tokenizer.decode(trocr_out[0].cpu(), skip_special_tokens=True)
    #    print("Detected Text: {}".format(rec_text))
    
    # Check for the 'Enter' key press to exit the loop
    if cv2.waitKey(10) == 13:
        break
    
    #sleep(1)
    
cv2.destroyAllWindows()
cam.release()

In [ ]:
#cv2.destroyAllWindows()
#cam.release()

In [1]:
import sympy as sp
from sympy import *

t, x = sp.symbols("t x")
func = sp.sqrt(1 - t**2)

In [5]:
def imp(n):
    if n == 0:
        return 1
    k = Symbol("k")
    R = 1
    for k in range(1, n + 1):
        r = 2 * k - 1
        R = R * r
    return R

In [2]:
def imp1(n):
    s = []
    if n == 0:
        return 1
    k = Symbol("k")
    R = 1
    for k in range(1, n + 1):
        r = 2 * k - 1
        s.append(r)
        R = R * r
    return R, s

In [8]:
def c(n):
    coef = []
    k = Symbol("k")
    r = 0
    for i in range(1, n + 1):
        coef.append(imp(i - 1) / ((2**i) * (sp.factorial(i)) * (2 * i + 1)))
        r = r + (imp(i - 1) / ((2**i) * (sp.factorial(i)) * (2 * i + 1)))
        
        
        
    return r, coef, (4 * (1 - r))

In [12]:
imp1(10)

(654729075, [1, 3, 5, 7, 9, 11, 13, 15, 17, 19])

In [16]:
c(50)

(3736924683489064781307803201129988157774515668671853790902116489687409/17434548098274066993372956221607376190021791263028615338664768464486400,
 [1/6,
  1/40,
  1/112,
  5/1152,
  7/2816,
  21/13312,
  11/10240,
  429/557056,
  715/1245184,
  2431/5505024,
  4199/12058624,
  29393/104857600,
  52003/226492416,
  185725/973078528,
  334305/2080374784,
  3231615/23622320128,
  3535767/30064771072,
  64822395/635655159808,
  39803225/446676598784,
  883631595/11269994184704,
  1641030105/23639499997184,
  407771117/6597069766656,
  11435320455/206708186021888,
  171529806825/3448068464705536,
  107492012277/2392537302040576,
  1215486600363/29836347531329536,
  2295919134019/61924494876344320,
  17383387729001/513410357520236544,
  32968493968795/1062849512059437056,
  125280277081421/4395513236313604096,
  34062379482967/1297036692682702848,
  14544636039226909/599519182395560427520,
  27767032438524099/1235931852938539958272,
  35389355068707185/1697100454781278748672,
  20323601053743

In [17]:
13697623414785002212065153020477388032247275594356761547762651974798991 / 4358637024568516748343239055401844047505447815757153834666192116121600

3.142639163934739

In [18]:
from pprint import pprint

In [19]:
n = 20
taylor_serie = sp.series(func, t, 0, n)
integral = sp.integrate(taylor_serie, (t, 0, x))
integral = integral.removeO()


-715*x**19/1245184 - 429*x**17/557056 - 11*x**15/10240 - 21*x**13/13312 - 7*x**11/2816 - 5*x**9/1152 - x**7/112 - x**5/40 - x**3/6 + x


In [21]:
integral

-715*x**19/1245184 - 429*x**17/557056 - 11*x**15/10240 - 21*x**13/13312 - 7*x**11/2816 - 5*x**9/1152 - x**7/112 - x**5/40 - x**3/6 + x